In [ ]:
import os

%cd /home1/kelidari/

REPO_URL = f"https://github.com/Nikelroid/multimodal-sentiment-classification/"
REPO_NAME = "multimodal-sentiment-classification"

if not os.path.exists(REPO_NAME):
    print("\nCloning repository...")
    res = os.system(f"git clone {REPO_URL} {REPO_NAME} > /dev/null 2>&1")
    if res == 0:
        print("Successfully cloned!")
    %cd $REPO_NAME
else:
    print("\nRepository found. Pulling latest changes...")
    %cd $REPO_NAME
    os.system(f"git remote set-url origin {REPO_URL}")
    os.system("git pull")
    print("Done.")


In [ ]:
import os
import sys

miniconda_path = f"{os.environ.get('HOME', '')}/miniconda/bin"
os.environ["PATH"] = f"{miniconda_path}:" + os.environ.get("PATH", "")

print(f"Conda PATH securely bound to: {miniconda_path}")


In [ ]:
import os

# Preemptively accept Conda TOS to prevent hanging prompts
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true

# Intelligently handle environment creation or update
env_name = "multimodal_env"
print("Checking environment status...")
res = os.system(f"conda env list | grep {env_name} > /dev/null")
if res == 0:
    print(f"{env_name} exists! Synchronizing any missing packages ...")
    !conda env update -f environment.yml --prune
else:
    print(f"{env_name} does not exist. Creating fresh environment...")
    !conda env create -f environment.yml


In [ ]:
DATA_PATH = "data"
DATASET_NAME = "MSCTD"
MODEL = "roberta-base"
BATCH_SIZE = 32
TASK = "classification"


## 1. Data Ingestion
Download the exact multimodal datasets directly via Conda.

In [ ]:
!conda run -n multimodal_env python -u src/data/ingestion.py --data_dir {DATA_PATH}


## 2. Test Functionality
Write a quick test script to verify model initialization and forward pass works, then run it using conda to make sure architectures perform functionally.

In [ ]:
%%writefile test_func.py
import torch
from src.config import TEXT_MODEL_NAME, VISION_BACKBONE_NAME
from transformers import AutoTokenizer, AutoImageProcessor
from src.models.multimodal import MultimodalFusionNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Testing Architecture natively on {device}...")

test_model = MultimodalFusionNet(
    text_model_name=TEXT_MODEL_NAME,
    vit_model_name=VISION_BACKBONE_NAME,
    use_audio=True
).to(device)

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
test_txt = tokenizer(["Testing infrastructure parameters."], padding=True, truncation=True, return_tensors="pt").to(device)
test_img = torch.randn(1, 3, 224, 224).to(device)
test_aud = torch.randn(1, 16000).to(device)

with torch.no_grad():
    test_logits = test_model(test_txt['input_ids'], test_txt['attention_mask'], test_img, test_aud)
    
assert test_logits.shape == (1, 3), "Functional Test Failed: Dimensions deviated."
print("Functionality Test Passed! Architecture processes standard tensor flows impeccably.")


In [ ]:
!conda run -n multimodal_env python -u test_func.py


## 3. Train Pipeline
Run the main multimodal training pipeline directly utilizing the conda environment. This natively connects to wandb.

In [ ]:
!conda run -n multimodal_env python -u src/pipelines/train.py --task {TASK} --data_dir {DATA_PATH} --dataset_name {DATASET_NAME} --model_name {MODEL} --batch_size {BATCH_SIZE}


## 4. Evaluation Pipeline
Run the independent evaluation loop to fetch the test dataset dataloaders and `best_multimodal.pt` weights to calculate Accuracy, Precision, Recall, F1.

In [ ]:
!conda run -n multimodal_env python -u src/pipelines/evaluate.py --task {TASK} --data_dir {DATA_PATH} --dataset_name {DATASET_NAME} --model_name {MODEL} --batch_size {BATCH_SIZE}
